# Assignment 4 — Training a DCGAN on Fashion-MNIST

**Objective:** Understand how a Deep Convolutional Generative Adversarial Network (DCGAN)
is trained, and how the Generator and Discriminator improve (and compete) over time.

**Framework:** PyTorch

**What this notebook does**
1. Loads the Fashion-MNIST dataset.
2. Builds a Generator (transposed convolutions) and a Discriminator (convolutions).
3. Trains both networks adversarially for 30 epochs.
4. Saves a 4×4 grid of generated images every 5 epochs.
5. Records Generator loss and Discriminator loss for every batch/epoch.
6. Plots the loss curves and displays the saved image grids side by side.
7. Ends with a written analysis of the training dynamics.

> Run this notebook top to bottom. A GPU (CUDA) is strongly recommended — on CPU, 30
> epochs will be slow. In Google Colab: **Runtime → Change runtime type → GPU**.


## 1. Imports and Setup

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.utils as vutils
from torch.utils.data import DataLoader

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Output folders
os.makedirs("outputs/images", exist_ok=True)
os.makedirs("outputs/checkpoints", exist_ok=True)


## 2. Hyperparameters

In [ ]:
IMAGE_SIZE   = 32      # Fashion-MNIST images (28x28) are resized to 32x32
NC           = 1       # number of channels (grayscale)
NZ           = 100     # size of the latent (noise) vector
NGF          = 64      # base number of generator feature maps
NDF          = 64      # base number of discriminator feature maps
BATCH_SIZE   = 128
NUM_EPOCHS   = 30      # >= 25-30 as required
LR           = 2e-4    # standard DCGAN learning rate
BETA1        = 0.5     # Adam beta1 (standard DCGAN choice)
SAVE_EVERY   = 5       # save a 4x4 grid every 5 epochs
GRID_N       = 4       # 4 x 4 grid


## 3. Dataset — Fashion-MNIST

Images are resized from 28×28 to 32×32 (so that the conv/deconv stride-2 stack divides
evenly: 32 → 16 → 8 → 4 → 1) and normalized to the range **[-1, 1]** to match the
generator's `Tanh` output.

In [ ]:
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))   # -> pixel range [-1, 1]
])

train_dataset = torchvision.datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)

dataloader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, drop_last=True
)

print(f"Number of training images: {len(train_dataset)}")
print(f"Number of batches per epoch: {len(dataloader)}")

# Sanity check: visualize a batch of real images
real_batch = next(iter(dataloader))
plt.figure(figsize=(6, 6))
plt.axis("off")
plt.title("Sample Real Fashion-MNIST Images")
plt.imshow(np.transpose(
    vutils.make_grid(real_batch[0][:16], nrow=4, padding=2, normalize=True).cpu(),
    (1, 2, 0)))
plt.show()


## 4. Weight Initialization

Following the original DCGAN paper (Radford et al., 2015): weights are initialized from
a Normal distribution with mean 0 and std 0.02.

In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


## 5. Generator (Transposed Convolutions)

Maps a latent vector `z` of shape `(NZ, 1, 1)` to a `32x32x1` image.

`z (100,1,1) -> 256x4x4 -> 128x8x8 -> 64x16x16 -> 1x32x32`

In [ ]:
class Generator(nn.Module):
    def __init__(self, nz=NZ, ngf=NGF, nc=NC):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            # input: nz x 1 x 1  ->  (ngf*4) x 4 x 4
            nn.ConvTranspose2d(nz, ngf * 4, kernel_size=4, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),

            # (ngf*4) x 4 x 4  ->  (ngf*2) x 8 x 8
            nn.ConvTranspose2d(ngf * 4, ngf * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),

            # (ngf*2) x 8 x 8  ->  ngf x 16 x 16
            nn.ConvTranspose2d(ngf * 2, ngf, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),

            # ngf x 16 x 16  ->  nc x 32 x 32
            nn.ConvTranspose2d(ngf, nc, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        return self.main(z)


## 6. Discriminator (Convolutions)

Maps a `32x32x1` image to a single scalar (real/fake probability).

`1x32x32 -> 64x16x16 -> 128x8x8 -> 256x4x4 -> 1x1x1`

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, ndf=NDF, nc=NC):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            # nc x 32 x 32  ->  ndf x 16 x 16
            nn.Conv2d(nc, ndf, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # ndf x 16 x 16  ->  (ndf*2) x 8 x 8
            nn.Conv2d(ndf, ndf * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # (ndf*2) x 8 x 8  ->  (ndf*4) x 4 x 4
            nn.Conv2d(ndf * 2, ndf * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),

            # (ndf*4) x 4 x 4  ->  1 x 1 x 1
            nn.Conv2d(ndf * 4, 1, kernel_size=4, stride=1, padding=0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, img):
        return self.main(img).view(-1, 1).squeeze(1)


## 7. Instantiate Models, Loss, and Optimizers

In [ ]:
netG = Generator().to(device)
netG.apply(weights_init)

netD = Discriminator().to(device)
netD.apply(weights_init)

print(netG)
print(netD)

criterion = nn.BCELoss()

# Fixed noise vector used to visualize how the SAME latent codes evolve over training
fixed_noise = torch.randn(GRID_N * GRID_N, NZ, 1, 1, device=device)

REAL_LABEL = 1.0
FAKE_LABEL = 0.0

optimizerD = optim.Adam(netD.parameters(), lr=LR, betas=(BETA1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=LR, betas=(BETA1, 0.999))


## 8. Helper: Save a 4×4 Grid of Generated Images

In [ ]:
def save_image_grid(generator, epoch, noise, out_dir="outputs/images"):
    generator.eval()
    with torch.no_grad():
        fake = generator(noise).detach().cpu()
    grid = vutils.make_grid(fake, nrow=GRID_N, padding=2, normalize=True)

    fig = plt.figure(figsize=(4, 4))
    plt.axis("off")
    plt.title(f"Generated Images — Epoch {epoch}")
    plt.imshow(np.transpose(grid, (1, 2, 0)), cmap="gray")
    save_path = os.path.join(out_dir, f"epoch_{epoch:03d}.png")
    plt.savefig(save_path, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    generator.train()
    return save_path


## 9. Training Loop

Standard DCGAN training step for each batch:

1. **Discriminator step:** maximize `log(D(real)) + log(1 - D(G(z)))`
   - Forward real images through D, compute loss vs. label=1
   - Forward fake images (from G) through D, compute loss vs. label=0
   - Backprop and update D
2. **Generator step:** maximize `log(D(G(z)))` (non-saturating trick)
   - Forward the same fake images through the *updated* D, compute loss vs. label=1
   - Backprop and update G only

Losses are recorded **every batch** (for a fine-grained curve) and averaged **per epoch**
(for a smoother trend line). A 4×4 grid from the fixed noise vector is saved every 5
epochs, so we can literally watch the same 16 latent codes evolve into images.

In [ ]:
G_losses_batch = []
D_losses_batch = []
G_losses_epoch = []
D_losses_epoch = []
saved_grid_paths = {}

start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_g_loss, epoch_d_loss = 0.0, 0.0

    for i, (real_images, _) in enumerate(dataloader):
        b_size = real_images.size(0)
        real_images = real_images.to(device)

        # =====================================================
        # (1) Update Discriminator: maximize log(D(x)) + log(1 - D(G(z)))
        # =====================================================
        netD.zero_grad()

        # -- real batch --
        label = torch.full((b_size,), REAL_LABEL, dtype=torch.float, device=device)
        output_real = netD(real_images)
        lossD_real = criterion(output_real, label)
        lossD_real.backward()

        # -- fake batch --
        noise = torch.randn(b_size, NZ, 1, 1, device=device)
        fake_images = netG(noise)
        label.fill_(FAKE_LABEL)
        output_fake = netD(fake_images.detach())
        lossD_fake = criterion(output_fake, label)
        lossD_fake.backward()

        lossD = lossD_real + lossD_fake
        optimizerD.step()

        # =====================================================
        # (2) Update Generator: maximize log(D(G(z)))
        # =====================================================
        netG.zero_grad()
        label.fill_(REAL_LABEL)   # generator wants D to think these are real
        output = netD(fake_images)
        lossG = criterion(output, label)
        lossG.backward()
        optimizerG.step()

        # -- bookkeeping --
        G_losses_batch.append(lossG.item())
        D_losses_batch.append(lossD.item())
        epoch_g_loss += lossG.item()
        epoch_d_loss += lossD.item()

        if i % 100 == 0:
            print(f"[Epoch {epoch}/{NUM_EPOCHS}] [Batch {i}/{len(dataloader)}] "
                  f"D_loss: {lossD.item():.4f}  G_loss: {lossG.item():.4f}")

    avg_g = epoch_g_loss / len(dataloader)
    avg_d = epoch_d_loss / len(dataloader)
    G_losses_epoch.append(avg_g)
    D_losses_epoch.append(avg_d)
    elapsed = time.time() - start_time
    print(f"==> Epoch {epoch} done. Avg D_loss: {avg_d:.4f}  Avg G_loss: {avg_g:.4f}  "
          f"(elapsed: {elapsed/60:.1f} min)")

    # Save a 4x4 grid every 5 epochs (and always on the first/last epoch)
    if epoch % SAVE_EVERY == 0 or epoch == 1 or epoch == NUM_EPOCHS:
        path = save_image_grid(netG, epoch, fixed_noise)
        saved_grid_paths[epoch] = path

# Save final model checkpoints
torch.save(netG.state_dict(), "outputs/checkpoints/generator_final.pth")
torch.save(netD.state_dict(), "outputs/checkpoints/discriminator_final.pth")
print("Training complete. Models saved to outputs/checkpoints/")


## 10. Plot Generator and Discriminator Loss Curves

In [ ]:
# Per-batch loss curve (fine-grained, shows adversarial oscillation)
plt.figure(figsize=(10, 5))
plt.title("Generator and Discriminator Loss During Training (per batch)")
plt.plot(G_losses_batch, label="Generator", alpha=0.8)
plt.plot(D_losses_batch, label="Discriminator", alpha=0.8)
plt.xlabel("Training Iteration (batch)")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("outputs/loss_curve_per_batch.png", bbox_inches="tight")
plt.show()


In [ ]:
# Per-epoch averaged loss curve (smoother trend)
plt.figure(figsize=(10, 5))
plt.title("Average Generator and Discriminator Loss per Epoch")
plt.plot(range(1, NUM_EPOCHS + 1), G_losses_epoch, marker="o", label="Generator")
plt.plot(range(1, NUM_EPOCHS + 1), D_losses_epoch, marker="o", label="Discriminator")
plt.xlabel("Epoch")
plt.ylabel("Average Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("outputs/loss_curve_per_epoch.png", bbox_inches="tight")
plt.show()


## 11. Compare Generated Image Grids Across Epochs

In [ ]:
epochs_to_show = sorted(saved_grid_paths.keys())
n = len(epochs_to_show)
cols = min(n, 4)
rows = int(np.ceil(n / cols))

fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
axes = np.array(axes).reshape(-1)

for ax, epoch in zip(axes, epochs_to_show):
    img = plt.imread(saved_grid_paths[epoch])
    ax.imshow(img)
    ax.set_title(f"Epoch {epoch}")
    ax.axis("off")

# hide any unused subplots
for ax in axes[len(epochs_to_show):]:
    ax.axis("off")

plt.tight_layout()
plt.savefig("outputs/images/epoch_comparison_grid.png", bbox_inches="tight")
plt.show()


## 12. Written Observations — Training Process

*(Fill in / adjust the specifics — exact epoch numbers, loss values — based on the run
you actually produced above. The structure below reflects the pattern that is typical
for a correctly-training DCGAN on Fashion-MNIST, which you should compare your results
against.)*

**Early epochs (≈1–5):**
The generator's output is close to random noise/static — blotches of gray with no clear
structure. `D_loss` is usually very low here because the discriminator finds it trivial
to separate obvious noise from real clothing images, while `G_loss` is high (the
generator is easily "caught"). The image grid saved at epoch 1 typically shows only
vague blob-like silhouettes with no identifiable garment shape.

**Middle epochs (≈10–15):**
Broad shape and contrast start to emerge — you can begin to see rough outlines that
resemble general clothing silhouettes (e.g. a rectangular shirt-like shape vs. an
elongated trouser-like shape vs. a shoe-like shape), though textures are still blurry
and edges are soft. This is usually the stage where recognisable clothing categories
(t-shirt, trouser, sneaker, bag) first become distinguishable by eye, even if fine
detail (fabric texture, logos, straps) is still missing. `D_loss` and `G_loss` begin to
oscillate around each other rather than one dominating, which is a sign that the
adversarial game is becoming balanced.

**Later epochs (≈20–30):**
Image sharpness and category consistency continue to improve, though gains per epoch
become smaller — a sign of diminishing returns as the generator approaches the limit of
what this relatively small (32×32, single-channel) DCGAN architecture can represent.
By around epoch 25–30 the generated grid should show fairly consistent, recognisable
Fashion-MNIST-style garments with plausible silhouettes, though some samples may still
look blurry or show minor artifacts (holes, asymmetries, blended edges between
categories).

**Does quality plateau?**
Yes — after roughly epoch 20–25 the *visual* improvement from one saved grid to the next
becomes noticeably smaller, even though the loss values keep oscillating. This is
expected: DCGAN loss curves do not monotonically decrease toward zero the way a
supervised loss does, because the discriminator and generator are locked in a
zero-sum/minimax game — a lower loss for one directly means a harder time for the other.
A stable-looking oscillation around a roughly constant band (rather than one loss
collapsing to ~0 while the other explodes) is the healthy sign that training has reached
a workable equilibrium, not necessarily that images have stopped improving completely.

**Mode collapse / instability check:**
Inspect the 4×4 grids: if many of the 16 generated images look nearly identical (same
garment type, same pose/orientation) despite starting from 16 *different* fixed noise
vectors, that is a sign of (partial) mode collapse. If the loss curves show `D_loss`
suddenly dropping to ~0 while `G_loss` spikes and stays high, that indicates the
discriminator has "won" and stopped providing a useful learning signal to the generator
— training has become unstable in the discriminator's favor. Occasional sharp spikes in
`G_loss` or `D_loss` on the per-batch curve are normal (a side-effect of BCE loss and
minibatch noise); persistent divergence (one loss trending to 0 and staying there for
many consecutive epochs) is the real red flag to watch for.

**Summary:** the DCGAN progression on Fashion-MNIST typically follows: *random noise →
vague blobs → rough garment silhouettes → recognisable but blurry clothing → sharper,
mostly-consistent garments*, with the steepest visual improvement happening in the first
~15 epochs and diminishing (but still non-zero) gains afterward, while the adversarial
loss curves oscillate rather than converge — which is the expected behavior for GAN
training rather than a bug.
